# 05 · Estacionalidad descriptiva (P6)

**Objetivo:** identificar qué productos muestran un patrón estacional estable de precios
**Preguntas que responde:** P6
**Entrada:** `data/processed/precios_limpio.parquet`
**Salida:** `images/p6_estacionalidad.png` (en la siguiente tanda)
**Decisiones tomadas:** D-07, D-18 (se cierra aquí)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("../data/processed/precios_limpio.parquet")

MIN_OBS = 5          # filas mínimas por mes y serie
MIN_MESES_ANIO = 6   # meses con dato para que una serie-año entre
MIN_MERCADOS = 3     # mercados mínimos por producto, año y mes

b = df[df["anio"] <= 2025].copy()
for c in ["producto", "mercado", "unidad"]:
    b[c] = b[c].astype(str)
b["mes"] = b["fecha"].dt.month

mu = (b.groupby(["producto", "mercado", "unidad", "anio", "mes"])
        .agg(precio=("precio_prom", "median"), n=("precio_prom", "size")).reset_index())
mu = mu[mu["n"] >= MIN_OBS]

# una sola etiqueta por (producto, mercado): la de más meses
meses = mu.groupby(["producto", "mercado", "unidad"]).size().rename("meses").reset_index()
dom = meses.sort_values("meses", ascending=False).drop_duplicates(["producto", "mercado"])
s = mu.merge(dom[["producto", "mercado", "unidad"]], on=["producto", "mercado", "unidad"])
s["lp"] = np.log(s["precio"])

g = s.groupby(["producto", "mercado", "anio"])["lp"]
s["k"] = g.transform("size")
s["dev"] = s["lp"] - g.transform("mean")
s = s[s["k"] >= MIN_MESES_ANIO]

prof = (s.groupby(["producto", "anio", "mes"])
          .agg(dev=("dev", "median"), mercados=("mercado", "nunique")).reset_index())
prof = prof[prof["mercados"] >= MIN_MERCADOS]

print("series-año:", s.groupby(["producto", "mercado", "anio"]).ngroups,
      "| productos:", s["producto"].nunique(),
      "| celdas producto-año-mes con 3+ mercados:", len(prof),
      "| productos:", prof["producto"].nunique())

series-año: 2227 | productos: 69 | celdas producto-año-mes con 3+ mercados: 2897 | productos: 53


In [2]:
def estabilidad(M, min_meses=5):
    cors = []
    for y in M.index:
        otros = M.drop(index=y).mean(axis=0, skipna=True)
        x = M.loc[y]
        ok = x.notna() & otros.notna()
        if ok.sum() >= min_meses and x[ok].std() > 0 and otros[ok].std() > 0:
            cors.append(np.corrcoef(x[ok], otros[ok])[0, 1])
    return (float(np.mean(cors)), len(cors)) if len(cors) >= 3 else (np.nan, len(cors))

filas, mats = [], {}
for p, d in prof.groupby("producto"):
    M = d.pivot(index="anio", columns="mes", values="dev")
    perfil = M.mean(axis=0, skipna=True).dropna()
    est, nanios = estabilidad(M)
    if len(perfil) < 6 or np.isnan(est):
        continue
    mats[p] = M
    filas.append(dict(producto=p, anios=nanios, meses=len(perfil), estabilidad=est,
                      amplitud_pct=(np.exp(perfil.max() - perfil.min()) - 1) * 100,
                      mes_pico=int(perfil.idxmax()), mes_valle=int(perfil.idxmin())))
res = pd.DataFrame(filas).sort_values("estabilidad", ascending=False).reset_index(drop=True)
print("productos evaluables:", len(res), "de", prof["producto"].nunique())

rng = np.random.default_rng(42)
nulos = []
for p, M in mats.items():
    for _ in range(100):
        Mp = M.copy()
        for y in Mp.index:
            v = Mp.loc[y].dropna().values
            Mp.loc[y, Mp.loc[y].notna()] = rng.permutation(v)
        e, _ = estabilidad(Mp)
        if not np.isnan(e):
            nulos.append(e)
q95 = float(np.percentile(nulos, 95))
print("estabilidad esperada por azar: mediana", round(float(np.median(nulos)), 2),
      "| percentil 95:", round(q95, 2))
for u in [0.5, 0.7, q95]:
    print(f"estabilidad >= {u:.2f}:", int((res["estabilidad"] >= u).sum()))
print(res.round(2).to_string(index=False))

productos evaluables: 44 de 53
estabilidad esperada por azar: mediana 0.01 | percentil 95: 0.31
estabilidad >= 0.50: 26
estabilidad >= 0.70: 14
estabilidad >= 0.31: 37
        producto  anios  meses  estabilidad  amplitud_pct  mes_pico  mes_valle
           Limón      6     12         0.98        378.13         3          8
 Arándano (blue)      4      9         0.95        275.69         9          2
        Frutilla      6     12         0.95        157.98         8         11
  Poroto granado      4      6         0.95         36.61        12          3
         Manzana      6     12         0.92         71.48         1          7
            Pera      6     12         0.91         56.12        12          5
       Alcachofa      6      8         0.91         95.00         6         10
            Haba      6      9         0.89        116.97         5         10
         Naranja      6     12         0.88        117.78         4          8
         Zapallo      6     12         0.8